# IPL Matches EDA Improved
This notebook analyzes IPL match and ball-by-ball data from 2008 to 2022 using cleaner preprocessing, improved visualizations, and a predictive model for match outcomes.

## Setup and imports
Install packages if needed and import required libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_style('darkgrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'figure.facecolor': 'white'})

## Load data
The notebook assumes the CSV files are available in the parent folder relative to this notebook.

In [ ]:
matches_path = '../IPL_Matches_2008_2022.csv'
balls_path = '../IPL_Ball_by_Ball_2008_2022.csv'

matches = pd.read_csv(matches_path)
balls = pd.read_csv(balls_path)

matches.head()

## Data cleaning and preparation
Clean the matches dataset and standardize team names.

In [ ]:
matches['Date'] = pd.to_datetime(matches['Date'], errors='coerce')
matches['SuperOver'] = matches['SuperOver'].fillna('NoResult')
matches['WinningTeam'] = matches['WinningTeam'].fillna('NoResult')
matches['Player_of_Match'] = matches['Player_of_Match'].fillna('NoResult')
matches['Margin'] = matches['Margin'].fillna('NoMargin')

rename_map = {
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['Team1', 'Team2', 'WinningTeam', 'TossWinner']:
    matches[col] = matches[col].replace(rename_map)

matches['Season'] = pd.DatetimeIndex(matches['Date']).year
matches['Team1Wins'] = (matches['WinningTeam'] == matches['Team1']).astype(int)
matches['TossWinnerIsTeam1'] = (matches['TossWinner'] == matches['Team1']).astype(int)

matches.info()

## Match-level EDA
Visualize the most important summary statistics from match data.

In [ ]:
match_counts = pd.concat([matches['Team1'], matches['Team2']]).value_counts().reset_index()
match_counts.columns = ['Team', 'MatchesPlayed']
fig = px.bar(match_counts.sort_values('MatchesPlayed', ascending=False), x='MatchesPlayed', y='Team', orientation='h',
             title='IPL Matches Played per Team', color='MatchesPlayed', color_continuous_scale='viridis')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
win_counts = matches['WinningTeam'].value_counts().reset_index()
win_counts.columns = ['Team', 'Wins']
summary = match_counts.merge(win_counts, on='Team', how='left').fillna(0)
summary['WinRatio'] = (summary['Wins'] / summary['MatchesPlayed']).round(3) * 100
fig = px.scatter(summary.sort_values('WinRatio', ascending=False).head(12), x='WinRatio', y='Team', size='MatchesPlayed',
                 title='Top Teams by Win Ratio', color='WinRatio', color_continuous_scale='blues')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Season and venue analysis
New charts show match counts by season and top IPL venues.

In [ ]:
season_counts = matches.groupby('Season').size().reset_index(name='Matches')
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=season_counts, x='Season', y='Matches', palette='viridis', ax=ax)
ax.set_title('Total Matches per Season')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
venue_counts = matches['Venue'].value_counts().reset_index().head(12)
venue_counts.columns = ['Venue', 'HostedMatches']
fig = px.bar(venue_counts, x='HostedMatches', y='Venue', orientation='h',
             title='Top Venues by Matches Hosted', color='HostedMatches',
             color_continuous_scale='plasma')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Toss and match-outcome insights
Analyze how toss decisions relate to winning matches.

In [ ]:
toss_decision = matches['TossDecision'].value_counts().reset_index()
toss_decision.columns = ['Decision', 'Count']
fig = px.pie(toss_decision, names='Decision', values='Count', title='Toss Decisions in IPL')
fig.show()

In [ ]:
toss_win_rate = matches.assign(TossWinMatch=(matches['TossWinner'] == matches['WinningTeam'])).groupby('TossDecision')['TossWinMatch'].mean().reset_index()
toss_win_rate['TossWinPercent'] = toss_win_rate['TossWinMatch'] * 100
fig = px.bar(toss_win_rate, x='TossDecision', y='TossWinPercent', color='TossWinPercent',
             title='Match Win Rate by Toss Decision', labels={'TossWinPercent':'Win %'})
fig.show()

## Top players and performance metrics
Add new analysis for top players and bowlers.

In [ ]:
top_players = matches['Player_of_Match'].value_counts().reset_index().head(10)
top_players.columns = ['Player', 'Awards']
fig = px.bar(top_players, x='Awards', y='Player', orientation='h',
             title='Top 10 Player of the Match Winners', color='Awards', color_continuous_scale='magma')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
bowler_wickets = balls.groupby('bowler')['isWicketDelivery'].sum().reset_index()
top_bowlers = bowler_wickets.sort_values('isWicketDelivery', ascending=False).head(10)
fig = px.bar(top_bowlers, x='isWicketDelivery', y='bowler', orientation='h',
             title='Top 10 Wicket-Taking Bowlers', color='isWicketDelivery',
             color_continuous_scale='cividis')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## Predictive modeling: will Team 1 win?
Build a logistic regression model using toss, venue, and team data.

In [ ]:
model_data = matches[['Team1', 'Team2', 'Venue', 'TossDecision', 'TossWinnerIsTeam1', 'Team1Wins']].copy()
model_data = model_data.dropna()
X = model_data[['Team1', 'Team2', 'Venue', 'TossDecision', 'TossWinnerIsTeam1']]
y = model_data['Team1Wins']

categorical_features = ['Team1', 'Team2', 'Venue', 'TossDecision']
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_features),
    ('passthrough', 'passthrough', ['TossWinnerIsTeam1'])
])
model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, predictions))
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))
print('Classification report:')
print(classification_report(y_test, predictions, zero_division=0))